# Calibrating bias correction for a realistic Texas summer pool temp

**Goal**: find a `bias_correction` mode + parameter setting in
`scripts/pool_physics.py` that produces a *realistic* equilibrium pool
temperature for a hot, humid Texas summer scenario:

- Day high air temp  ≈ 110 °F (43.3 °C)
- Night low air temp ≈  80 °F (26.7 °C)
- Humid (RH ~55% day / ~80% night — Waco/Central-TX summer, not desert-dry)
- Target: pool settles around **~90 °F (32.2 °C)**, which matches typical
  observed wave-pool behavior at BSR Waco in peak summer — hot, but not
  air-temperature-hot, because a 2 m deep pool has a lot of thermal mass and
  strong evaporative cooling once wind/agitation is accounted for.

This follows on from `penman_bias_correction_sanity_check.ipynb`, which
showed a **single time-step** snapshot of each bias-correction mode. That's
useful for isolating flux math, but a single step can't show whether a mode
actually reaches a *sensible equilibrium* over many days — this notebook
does that by running a full multi-day diurnal simulation to a repeating
daily cycle (steady periodic state) and reports the resulting pool temp.


## Setup

In [1]:
import math

from pool_physics import (
    pool_thermal_balance_step,
    CE_MULTIPLIER,
    EVAP_WIND_FLOOR_MS,
)

def C_to_F(c):
    return c * 9 / 5 + 32

def F_to_C(f):
    return (f - 32) * 5 / 9

print(f"Production defaults: CE_MULTIPLIER={CE_MULTIPLIER}, EVAP_WIND_FLOOR_MS={EVAP_WIND_FLOOR_MS} m/s")

Production defaults: CE_MULTIPLIER=1.25, EVAP_WIND_FLOOR_MS=1.5 m/s


## Texas humid-summer diurnal forcing

Builds hourly `T_air`, `RH`, `wind`, and `solar` from smooth diurnal curves
anchored to the day-high / night-low targets, rather than a single flat
value — a single daily-mean step (as used in the earlier sanity-check
notebook and in `pull_api_data.py`'s daily model) washes out the fact that
the hottest, calmest, sunniest *hours* are what drive peak pool warming.
Real Waco pool config values (`scripts/pools_config.json`) are used for
`depth_m`, `ground_temp_C`, and the derived bottom U-value.

In [2]:
def diurnal_forcing(hour,
                     day_high_C=F_to_C(110), night_low_C=F_to_C(80),
                     rh_day=55.0, rh_night=80.0,
                     wind_day=2.6, wind_night=1.4,
                     solar_peak_Wm2=980.0):
    """Smooth diurnal cycle for one hour-of-day (0-24, fractional OK)."""
    frac = (hour - 6.0) / 24.0
    T_air = ((day_high_C + night_low_C) / 2
             + (day_high_C - night_low_C) / 2 * math.cos(2 * math.pi * (frac - 0.5)))

    # RH tracks inversely with temperature (dry heat of the day, humid at night)
    rh_frac = (T_air - night_low_C) / (day_high_C - night_low_C)
    RH = rh_day + (rh_night - rh_day) * (1 - rh_frac)

    # Wind picks up slightly at midday, calmer overnight
    if 6.0 <= hour <= 18.0:
        wind = wind_night + (wind_day - wind_night) * max(0.0, math.sin(math.pi * (hour - 6) / 12))
    else:
        wind = wind_night

    # Daylight solar bell curve, 6 AM - 8 PM (long Texas summer day)
    if 6.0 <= hour <= 20.0:
        solar_Wm2 = solar_peak_Wm2 * math.sin(math.pi * (hour - 6) / 14.0)
    else:
        solar_Wm2 = 0.0
    solar_MJm2_day_equiv = solar_Wm2 * 86400.0 / 1.0e6  # units pool_thermal_balance_step expects

    return T_air, RH, wind, solar_MJm2_day_equiv


# Sanity-check the forcing curve at a few hours
for h in [3, 9, 15, 21]:
    T_air, RH, wind, solar = diurnal_forcing(h)
    print(f"hour={h:2d}  T_air={C_to_F(T_air):5.1f}F  RH={RH:4.1f}%  wind={wind:.1f} m/s  solar={solar:.1f} MJ/m2/day-equiv")


hour= 3  T_air= 84.4F  RH=76.3%  wind=1.4 m/s  solar=0.0 MJ/m2/day-equiv
hour= 9  T_air= 84.4F  RH=76.3%  wind=2.2 m/s  solar=52.8 MJ/m2/day-equiv
hour=15  T_air=105.6F  RH=58.7%  wind=2.2 m/s  solar=76.3 MJ/m2/day-equiv
hour=21  T_air=105.6F  RH=58.7%  wind=1.4 m/s  solar=0.0 MJ/m2/day-equiv


## Real Waco pool physical parameters

From `scripts/pools_config.json` — `depth_m`, `ground_temp_C`, and a bottom
U-value derived the same way `pull_api_data.py` derives it (concrete slab +
soil column in series).

In [3]:
K_CONCRETE   = 1.4      # W/m/K (matches pool_physics.py)
L_CONCRETE_M = 0.3048   # 1 ft slab

# Waco config (scripts/pools_config.json)
depth_m       = 2.0
ground_temp_C = 20.0
k_soil_Wm1K   = 1.5
soil_depth_m  = 2.0

R_concrete = L_CONCRETE_M / K_CONCRETE
R_soil     = soil_depth_m / k_soil_Wm1K
bottom_u_Wm2K = 1.0 / (R_concrete + R_soil)

print(f"depth_m={depth_m}  ground_temp_C={ground_temp_C}  bottom_u_Wm2K={bottom_u_Wm2K:.3f}")


depth_m=2.0  ground_temp_C=20.0  bottom_u_Wm2K=0.645


## Multi-day diurnal simulation harness

Steps `pool_thermal_balance_step` hourly (`dt_days = 1/24`) for `days` days
starting from an arbitrary initial pool temp, and returns the **last day's**
hourly pool-temp series once the simulation has settled into a repeating
daily cycle (checked via `drift_C`, the max difference between the last two
days' hourly series — near zero means steady periodic state).

In [4]:
def run_sim(mode, days=20, T_pool_start_C=30.0,
            ce_multiplier=None, evap_wind_floor_ms=None):
    T_pool = T_pool_start_C
    dt = 1.0 / 24.0
    prev_day, last_day = None, None

    for _ in range(days):
        hourly = []
        for h in range(24):
            T_air, RH, wind, solar = diurnal_forcing(h + 0.5)
            T_pool, fx = pool_thermal_balance_step(
                T_pool, T_air, RH, wind, solar,
                depth_m=depth_m, ground_temp_C=ground_temp_C, bottom_u_Wm2K=bottom_u_Wm2K,
                dt_days=dt, bias_correction=mode,
                ce_multiplier=ce_multiplier, evap_wind_floor_ms=evap_wind_floor_ms,
            )
            hourly.append(T_pool)
        prev_day, last_day = last_day, hourly

    drift_C = max(abs(a - b) for a, b in zip(last_day, prev_day)) if prev_day else float('nan')
    return {
        'hourly_C': last_day,
        'mean_C': sum(last_day) / len(last_day),
        'min_C': min(last_day),
        'max_C': max(last_day),
        'drift_C': drift_C,
    }


## Step 1 — baseline behavior of all four modes (production defaults)

Using the *current* `pool_physics.py` defaults (`CE_MULTIPLIER=1.25`,
`EVAP_WIND_FLOOR_MS=1.5`) under sustained Texas humid-summer forcing —
not just a single hot/calm snapshot.

In [5]:
print(f"{'mode':16s} {'mean':>8s} {'min':>8s} {'max':>8s} {'drift_C':>10s}")
for mode in ['none', 'evap_multiplier', 'wind_floor', 'penman']:
    r = run_sim(mode)
    print(f"{mode:16s} {C_to_F(r['mean_C']):7.1f}F {C_to_F(r['min_C']):7.1f}F {C_to_F(r['max_C']):7.1f}F {r['drift_C']:10.4f}")

print("\nTarget: mean ~90F, reflecting a large, well-agitated wave pool that")
print("stays much cooler than the 110F day-high air temp thanks to strong")
print("evaporative cooling.")


mode                 mean      min      max    drift_C
none               107.0F   105.4F   108.6F     0.0038
evap_multiplier    105.1F   103.5F   106.7F     0.0020
wind_floor         106.8F   105.1F   108.4F     0.0036
penman             122.1F   120.3F   123.7F     0.0876

Target: mean ~90F, reflecting a large, well-agitated wave pool that
stays much cooler than the 110F day-high air temp thanks to strong
evaporative cooling.


### Result

All four modes at their *default* parameters settle into an unrealistically
hot equilibrium (~103-107°F for `none`/`evap_multiplier`/`wind_floor`, and
`penman` is even hotter at ~121°F, consistent with the earlier notebook
finding that `penman` under-predicts cooling). None of these are close to
the ~90°F target — the baseline model (and the notebook's small `1.25x`/
`1.5 m/s` tuning, which was fit for a different, presumably milder scenario)
isn't strong enough evaporative cooling for a **110°F humid Texas
day**. `penman` remains the worst performer and still needs a separate fix
(see the previous notebook's "Suspected cause" section) — it is excluded
from further tuning here.

## Step 2 — sweep `evap_multiplier` and `wind_floor` magnitudes

Rather than guess, sweep both correction modes across a wide range of
magnitudes and see what it actually takes to reach ~90°F under this
forcing.

In [6]:
print("=== evap_multiplier (CE_MULTIPLIER) sweep ===")
print(f"{'CE_MULTIPLIER':>14s} {'mean':>8s} {'min':>8s} {'max':>8s}")
for cm in [1.25, 3, 5, 8, 10, 12, 16, 20]:
    r = run_sim('evap_multiplier', ce_multiplier=cm)
    print(f"{cm:14.2f} {C_to_F(r['mean_C']):7.1f}F {C_to_F(r['min_C']):7.1f}F {C_to_F(r['max_C']):7.1f}F")

print()
print("=== wind_floor (EVAP_WIND_FLOOR_MS) sweep ===")
print(f"{'wind_floor_ms':>14s} {'mean':>8s} {'min':>8s} {'max':>8s}")
for wf in [1.5, 3, 5, 8, 10, 12, 16, 20]:
    r = run_sim('wind_floor', evap_wind_floor_ms=wf)
    print(f"{wf:14.1f} {C_to_F(r['mean_C']):7.1f}F {C_to_F(r['min_C']):7.1f}F {C_to_F(r['max_C']):7.1f}F")


=== evap_multiplier (CE_MULTIPLIER) sweep ===
 CE_MULTIPLIER     mean      min      max
          1.25   105.1F   103.5F   106.7F
          3.00    97.6F    95.8F    99.2F
          5.00    93.6F    91.7F    95.4F
          8.00    90.7F    88.4F    92.6F
         10.00    89.5F    87.0F    91.6F
         12.00    88.6F    85.9F    90.9F
         16.00    87.5F    84.3F    90.2F
         20.00    86.7F    83.1F    89.8F

=== wind_floor (EVAP_WIND_FLOOR_MS) sweep ===
 wind_floor_ms     mean      min      max
           1.5   106.8F   105.1F   108.4F
           3.0   102.4F   100.5F   104.2F
           5.0    98.0F    96.0F    99.8F
           8.0    94.2F    92.0F    96.2F
          10.0    92.6F    90.3F    94.7F
          12.0    91.4F    89.0F    93.6F
          16.0    89.7F    87.1F    92.1F
          20.0    88.6F    85.7F    91.1F


### Result

Both modes can reach ~90°F, but only with parameter values **far beyond**
the notebook-derived defaults (`CE_MULTIPLIER=1.25`, `EVAP_WIND_FLOOR_MS=1.5`):

- `evap_multiplier` needs **CE_MULTIPLIER ≈ 8** (a 6-8x jump from 1.25) to
  land near 90°F.
- `wind_floor` needs **EVAP_WIND_FLOOR_MS ≈ 12-16 m/s** (vs. 1.5 m/s
  default) — i.e. treating the *whole pool surface* as if a steady
  12-16 m/s wind were blowing across it at all times.

Both of these are large extrapolations from the original notebook's
sensitivity sweep, which was fit against milder conditions. But they are
**not physically absurd** for this specific application: this is a **wave
pool**, not a placid backyard pool — a wave-generating machine constantly
agitates and aerates a large fraction of the surface, which is a much
stronger analogue to "high sustained wind" than the bulk-transfer
formula (calibrated for calm, undisturbed open water) assumes. A wind-floor
in the 12-16 m/s-equivalent range is a plausible stand-in for "surface
constantly disturbed by wave machinery," rather than natural wind alone.

## Step 3 — pick a specific calibration and confirm it holds up

In [7]:
chosen_mode = 'wind_floor'
chosen_wind_floor = 14.0

r = run_sim(chosen_mode, evap_wind_floor_ms=chosen_wind_floor)
print(f"mode={chosen_mode}  evap_wind_floor_ms={chosen_wind_floor}")
print(f"  mean = {C_to_F(r['mean_C']):.1f}F   min = {C_to_F(r['min_C']):.1f}F   max = {C_to_F(r['max_C']):.1f}F")
print(f"  drift_C (steady-state check) = {r['drift_C']:.5f}")
print()
print("Hourly pool temp over the final (settled) day:")
for h, t_c in enumerate(r['hourly_C']):
    marker = '  <-- night low' if h in (4, 5) else ('  <-- afternoon peak' if h in (15, 16) else '')
    print(f"  hour {h:2d}: {C_to_F(t_c):5.1f}F{marker}")


mode=wind_floor  evap_wind_floor_ms=14.0
  mean = 90.5F   min = 88.0F   max = 92.7F
  drift_C (steady-state check) = 0.00000

Hourly pool temp over the final (settled) day:
  hour  0:  91.6F
  hour  1:  91.1F
  hour  2:  90.6F
  hour  3:  90.0F
  hour  4:  89.4F  <-- night low
  hour  5:  88.8F  <-- night low
  hour  6:  88.3F
  hour  7:  88.0F
  hour  8:  88.0F
  hour  9:  88.1F
  hour 10:  88.4F
  hour 11:  88.8F
  hour 12:  89.4F
  hour 13:  90.1F
  hour 14:  90.7F
  hour 15:  91.4F  <-- afternoon peak
  hour 16:  92.0F  <-- afternoon peak
  hour 17:  92.4F
  hour 18:  92.7F
  hour 19:  92.7F
  hour 20:  92.6F
  hour 21:  92.5F
  hour 22:  92.3F
  hour 23:  92.0F


### Result

`wind_floor=14.0 m/s` settles into a steady daily cycle (`drift_C` ≈ 0)
averaging **~90°F**, ranging roughly **88-93°F** across the day — hot, but
meaningfully below the 110°F day-high / cooler than a placid pool would be,
and never dropping below the 80°F night-air floor. This matches the target
scenario description ("90 in humid 110 deg air, 80 nights").

## Conclusion / recommendation

- **Do not use `penman`** as currently implemented — it under-cools even
  worse than baseline and needs the fix identified in
  `penman_bias_correction_sanity_check.ipynb` before it's usable.
- **`evap_multiplier` (CE_MULTIPLIER≈8) or `wind_floor` (EVAP_WIND_FLOOR_MS≈14 m/s)**
  both reach a realistic ~90°F equilibrium for this scenario.
  `wind_floor` is preferred here because its physical interpretation (a
  wave-machine-agitated surface behaving like sustained high wind) maps
  more directly onto *why* a wave pool should evaporate/cool faster than
  the plain bulk-transfer formula assumes, vs. `evap_multiplier`'s more
  generic "just scale the coefficient" fit.
- These are **exploratory findings only** — `pool_physics.py`'s production
  defaults (`ACTIVE_BIAS_CORRECTION='none'`, `CE_MULTIPLIER=1.25`,
  `EVAP_WIND_FLOOR_MS=1.5`) have **not** been changed by this notebook.
  Flip `ACTIVE_BIAS_CORRECTION` (or pass `bias_correction=` explicitly at
  call sites) only after confirming this holds up against real observed
  pool-temperature data, not just a synthetic diurnal forcing curve.